In [1]:
# ============================================================
# MNIST FL / Sequential-DFL Code Based on 3-5-2026 Version
# MNIST only, update-level DP, MIA + Gradient Inversion Attacks
# ============================================================

import os, time, copy, random, warnings, math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
from tqdm import trange
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ============================================================
# USER SETTINGS
# ============================================================

DATASET_NAME = "MNIST"
FRAMEWORK = "DFL"          # "FL", "DFL", or "BOTH"

RUN_NODP = True
RUN_DP = True

SEED = 1
NUM_NODES = 100
ALPHA = 0.5

ROUNDS = 100
LOCAL_EPOCHS = 1
BATCH = 128
BASE_LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
THRESHOLD = 0.80

EPSILONS = [0.5, 1.0, 2.0, 3.0, 4.0]
DELTA = 1e-5
CLIP_NORM = 1.0

ATTACK_SAMPLES = 1000
SAVE_EVERY = 10
RUN_MIA_ATTACKS = True

# ------------------------------------------------------------
# GRADIENT-INVERSION ATTACKS
# ------------------------------------------------------------
RUN_GRADIENT_ATTACKS = True
GRAD_ATTACK_CLIENTS = [0, 10, 25]
GRAD_ATTACK_BATCH_SIZE = 1
GRAD_ATTACK_ITERS = 1000
GRAD_ATTACK_LR = 0.10
GRAD_ATTACK_SAVE_IMAGES = True

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = f"FINAL_MNIST_3MAY_{FRAMEWORK}_{NUM_NODES}NODES_{ROUNDS}ROUNDS"
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
GRAD_ATTACK_DIR = os.path.join(OUTPUT_DIR, "gradient_attacks")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(GRAD_ATTACK_DIR, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
print("Using device:", DEVICE)
print("Dataset:", DATASET_NAME)
print("Framework:", FRAMEWORK)
print("Output:", OUTPUT_DIR)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# MODEL
# ============================================================

class CNN_MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


def build_model():
    return CNN_MNIST().to(DEVICE)

# ============================================================
# DATA
# ============================================================

def load_data():
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_data = torchvision.datasets.MNIST(
        root="./data", train=True, download=True, transform=train_transform
    )
    test_data = torchvision.datasets.MNIST(
        root="./data", train=False, download=True, transform=test_transform
    )
    return train_data, test_data


def get_labels(dataset):
    return np.array(dataset.targets)


def dirichlet_split_noniid(labels, num_clients, alpha=0.5, seed=1):
    rng = np.random.default_rng(seed)
    n_classes = len(np.unique(labels))
    class_indices = [np.where(labels == y)[0] for y in range(n_classes)]
    client_indices = [[] for _ in range(num_clients)]

    for c in range(n_classes):
        idx_c = class_indices[c].copy()
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        proportions = np.array([
            p * (len(client_indices[i]) < len(labels) / num_clients)
            for i, p in enumerate(proportions)
        ])
        if proportions.sum() == 0:
            proportions = np.repeat(1.0 / num_clients, num_clients)
        else:
            proportions = proportions / proportions.sum()

        split_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        split_idx = np.split(idx_c, split_points)
        for client_id, part in enumerate(split_idx):
            client_indices[client_id].extend(part.tolist())

    for i in range(num_clients):
        rng.shuffle(client_indices[i])
    return client_indices


train_dataset, test_dataset = load_data()
labels = get_labels(train_dataset)
client_indices = dirichlet_split_noniid(labels, NUM_NODES, ALPHA, SEED)

client_loaders = [
    DataLoader(Subset(train_dataset, idxs), batch_size=BATCH, shuffle=True, num_workers=0)
    for idxs in client_indices
]

test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=0)

# ============================================================
# DATA DISTRIBUTION VISUALIZATION
# ============================================================

def plot_data_distribution(client_indices, labels, out_dir, framework_label="DFL"):
    counts = np.zeros((len(client_indices), 10), dtype=int)
    for node_id, idxs in enumerate(client_indices):
        if len(idxs) > 0:
            node_labels = labels[idxs]
            for c in range(10):
                counts[node_id, c] = int(np.sum(node_labels == c))

    fig, ax = plt.subplots(figsize=(16, 6))
    bottom = np.zeros(len(client_indices), dtype=int)
    x = np.arange(len(client_indices))

    for c in range(10):
        ax.bar(x, counts[:, c], bottom=bottom, label=f"C{c}", width=0.85)
        bottom += counts[:, c]

    ax.set_title(f"{framework_label} Data Distribution - MNIST (Dirichlet α={ALPHA}, Nodes={NUM_NODES})")
    ax.set_xlabel("Node")
    ax.set_ylabel("Number of samples")
    ax.set_xlim(-1, len(client_indices))
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.22),
        ncol=10,
        frameon=True
    )
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout(rect=[0, 0, 1, 0.90])

    out_path = os.path.join(out_dir, f"MNIST_data_distribution_{framework_label}_{NUM_NODES}nodes.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    counts_df = pd.DataFrame(counts, columns=[f"Class_{c}" for c in range(10)])
    counts_df.insert(0, "Node", np.arange(len(client_indices)))
    counts_csv = os.path.join(out_dir, f"MNIST_data_distribution_{framework_label}_{NUM_NODES}nodes.csv")
    counts_df.to_csv(counts_csv, index=False)

    print("Saved data distribution figure:", out_path)
    print("Saved data distribution CSV:", counts_csv)
    return out_path, counts_csv

plot_data_distribution(client_indices, labels, FIG_DIR, framework_label="FL_DFL")

# ============================================================
# METRICS AND UTILITIES
# ============================================================

loss_fn = nn.CrossEntropyLoss()


def get_vec(model):
    return torch.cat([p.detach().view(-1).cpu() for p in model.parameters()])


def set_vec(model, vec):
    pointer = 0
    for p in model.parameters():
        numel = p.numel()
        p.data.copy_(vec[pointer:pointer + numel].view_as(p).to(DEVICE))
        pointer += numel


def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e6, neginf=-1e6)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


def get_lr(round_idx):
    return BASE_LR


def clip_update(update, clip_norm):
    norm = torch.norm(update, p=2)
    if norm > clip_norm:
        update = update * (clip_norm / (norm + 1e-12))
    return update


def gaussian_sigma(epsilon, delta, clip_norm):
    return math.sqrt(2.0 * math.log(1.25 / delta)) * clip_norm / max(epsilon, 1e-12)


def rounds_to_threshold(accs, threshold):
    for i, a in enumerate(accs, start=1):
        if a >= threshold:
            return i
    return None


def stability_round(accs, window=5, tolerance=0.001):
    if len(accs) < window:
        return None
    for i in range(len(accs) - window + 1):
        chunk = accs[i:i + window]
        if max(chunk) - min(chunk) <= tolerance:
            return i + 1
    return None


def moving_average_padded(x, w=5):
    if len(x) < w:
        return x
    ma = np.convolve(np.array(x), np.ones(w) / w, mode="valid")
    out = [None] * len(x)
    for i, v in enumerate(ma):
        out[w - 1 + i] = float(v)
    return out

# ============================================================
# LOCAL TRAINING
# ============================================================

def train_one_client(base_model, loader, lr):
    model = copy.deepcopy(base_model).to(DEVICE)
    model.train()
    opt = optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    for _ in range(LOCAL_EPOCHS):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e6, neginf=-1e6)
            loss = loss_fn(logits, y)
            loss = torch.nan_to_num(loss, nan=50.0, posinf=50.0, neginf=50.0)
            loss.backward()
            # Optimization stability only, not the DP clipping mechanism.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            opt.step()
    return model

# ============================================================
# UPDATE-LEVEL DP MECHANISMS FROM 3-5-2026 CODE
# ============================================================

def apply_fl_dp(global_model, local_models, epsilon):
    global_vec = get_vec(global_model)
    clipped_updates = []

    for local_model in local_models:
        local_vec = get_vec(local_model)
        update = local_vec - global_vec
        clipped_updates.append(clip_update(update, CLIP_NORM))

    avg_update = torch.stack(clipped_updates, dim=0).mean(dim=0)
    sigma = gaussian_sigma(epsilon, DELTA, CLIP_NORM) / NUM_NODES
    noise = torch.normal(mean=0.0, std=sigma, size=avg_update.shape)

    new_vec = global_vec + avg_update + noise
    new_vec = torch.nan_to_num(new_vec, nan=0.0, posinf=1e6, neginf=-1e6)
    new_vec = torch.clamp(new_vec, min=-1e6, max=1e6)
    set_vec(global_model, new_vec)
    return global_model, torch.norm(noise, p=2).item()


def apply_dfl_dp(token_model, before_vec, after_vec, epsilon):
    update = after_vec - before_vec
    clipped_update = clip_update(update, CLIP_NORM)
    sigma = gaussian_sigma(epsilon, DELTA, CLIP_NORM) / NUM_NODES
    noise = torch.normal(mean=0.0, std=sigma, size=clipped_update.shape)

    new_vec = before_vec + clipped_update + noise
    new_vec = torch.nan_to_num(new_vec, nan=0.0, posinf=1e6, neginf=-1e6)
    new_vec = torch.clamp(new_vec, min=-1e6, max=1e6)
    set_vec(token_model, new_vec)
    return token_model, torch.norm(noise, p=2).item()


def random_cover_order(num_nodes, rng):
    # Sequential DFL: the token visits every node once per communication round.
    return rng.permutation(num_nodes).tolist()

# ============================================================
# MEMBERSHIP INFERENCE ATTACKS
# ============================================================

def compute_entropy(probs, eps=1e-12):
    probs = np.clip(probs, eps, 1.0)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), eps, None)
    return -np.sum(probs * np.log(probs), axis=1)


def collect_predictions(model, dataset, indices, batch_size=256):
    model.eval()
    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, num_workers=0)
    probs_all, labels_all, losses_all = [], [], []
    criterion = nn.CrossEntropyLoss(reduction="none")

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
            logits = torch.clamp(logits, min=-50.0, max=50.0)
            probs = torch.softmax(logits, dim=1)
            probs = torch.nan_to_num(probs, nan=1e-8, posinf=1.0, neginf=1e-8)
            probs = probs / probs.sum(dim=1, keepdim=True).clamp_min(1e-12)
            losses = criterion(logits, y)
            losses = torch.nan_to_num(losses, nan=50.0, posinf=50.0, neginf=0.0)
            losses = torch.clamp(losses, min=0.0, max=50.0)

            probs_all.append(probs.cpu().numpy())
            labels_all.append(y.cpu().numpy())
            losses_all.append(losses.cpu().numpy())

    return np.vstack(probs_all), np.concatenate(labels_all), np.concatenate(losses_all)


def build_attack_features(probs, labels, losses, mode):
    probs = np.nan_to_num(probs, nan=1e-8, posinf=1.0, neginf=1e-8)
    probs = np.clip(probs, 1e-8, 1.0)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)

    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    max_conf = sorted_probs[:, 0]
    second_conf = sorted_probs[:, 1]
    third_conf = sorted_probs[:, 2]
    margin = max_conf - second_conf
    true_conf = probs[np.arange(len(labels)), labels]
    entropy = compute_entropy(probs)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)

    if mode == "confidence":
        X = np.column_stack([
            probs, sorted_probs[:, :3], max_conf, second_conf, third_conf,
            margin, true_conf, entropy, losses, correct
        ])
    elif mode == "loss":
        X = np.column_stack([
            losses, probs, sorted_probs[:, :3], true_conf, max_conf,
            second_conf, third_conf, margin, entropy, correct
        ])
    elif mode == "entropy":
        X = np.column_stack([
            entropy, probs, sorted_probs[:, :3], max_conf, second_conf,
            third_conf, margin, true_conf, losses, correct
        ])
    else:
        raise ValueError("mode must be confidence, loss, or entropy")

    return X.astype(np.float32)


def train_mlp_attack(X, y, seed=1):
    X = np.nan_to_num(X, nan=0.0, posinf=50.0, neginf=-50.0)
    clf = MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        learning_rate_init=1e-3,
        max_iter=400,
        random_state=seed
    )
    clf.fit(X, y)
    return clf


def evaluate_attack(clf, X_member, X_nonmember):
    member_scores = clf.predict_proba(X_member)[:, 1]
    nonmember_scores = clf.predict_proba(X_nonmember)[:, 1]
    y_true = np.concatenate([np.ones(len(member_scores)), np.zeros(len(nonmember_scores))])
    y_score = np.concatenate([member_scores, nonmember_scores])
    y_pred = (y_score >= 0.5).astype(int)
    return (
        accuracy_score(y_true, y_pred),
        roc_auc_score(y_true, y_score),
        float(np.mean(member_scores) - np.mean(nonmember_scores))
    )


def run_three_attacks(model):
    if not RUN_MIA_ATTACKS:
        return {
            "Confidence Attack Accuracy": None, "Confidence Attack AUC": None, "Confidence Gap": None,
            "Loss Attack Accuracy": None, "Loss Attack AUC": None, "Loss Gap": None,
            "Entropy Attack Accuracy": None, "Entropy Attack AUC": None, "Entropy Gap": None
        }

    rng = np.random.default_rng(SEED)
    train_member_idx = rng.choice(len(train_dataset), size=min(ATTACK_SAMPLES, len(train_dataset)), replace=False)
    train_nonmember_idx = rng.choice(len(test_dataset), size=min(ATTACK_SAMPLES, len(test_dataset)), replace=False)
    eval_member_idx = rng.choice(len(train_dataset), size=min(ATTACK_SAMPLES, len(train_dataset)), replace=False)
    eval_nonmember_idx = rng.choice(len(test_dataset), size=min(ATTACK_SAMPLES, len(test_dataset)), replace=False)

    tm_p, tm_y, tm_l = collect_predictions(model, train_dataset, train_member_idx)
    tn_p, tn_y, tn_l = collect_predictions(model, test_dataset, train_nonmember_idx)
    em_p, em_y, em_l = collect_predictions(model, train_dataset, eval_member_idx)
    en_p, en_y, en_l = collect_predictions(model, test_dataset, eval_nonmember_idx)

    results = {}
    for mode, name in [("confidence", "Confidence"), ("loss", "Loss"), ("entropy", "Entropy")]:
        X_member_train = build_attack_features(tm_p, tm_y, tm_l, mode)
        X_nonmember_train = build_attack_features(tn_p, tn_y, tn_l, mode)
        X_train = np.vstack([X_member_train, X_nonmember_train])
        y_train = np.concatenate([np.ones(len(X_member_train)), np.zeros(len(X_nonmember_train))])

        clf = train_mlp_attack(X_train, y_train, seed=SEED + len(mode))
        X_member_eval = build_attack_features(em_p, em_y, em_l, mode)
        X_nonmember_eval = build_attack_features(en_p, en_y, en_l, mode)
        acc, auc, gap = evaluate_attack(clf, X_member_eval, X_nonmember_eval)

        results[f"{name} Attack Accuracy"] = acc
        results[f"{name} Attack AUC"] = auc
        results[f"{name} Gap"] = gap

        print(f"{name} MIA | Accuracy: {acc:.4f} | AUC: {auc:.4f} | Gap: {gap:.6f}")

    return results

# ============================================================
# GRADIENT-INVERSION ATTACKS: DLG / iDLG / invGrad
# ============================================================

def denormalize_tensor(x):
    x = x.detach().cpu().clone()
    mean = torch.tensor([0.1307]).view(1, 1, 1, 1)
    std = torch.tensor([0.3081]).view(1, 1, 1, 1)
    return torch.clamp(x * std + mean, 0.0, 1.0)


def tensor_mse(x_rec, x_true):
    xr = denormalize_tensor(x_rec)
    xt = denormalize_tensor(x_true)
    return float(torch.mean((xr - xt) ** 2).item())


def tensor_psnr(x_rec, x_true):
    mse = max(tensor_mse(x_rec, x_true), 1e-12)
    return float(10.0 * np.log10(1.0 / mse))


def tensor_ssim_simple(x_rec, x_true):
    xr = denormalize_tensor(x_rec).view(x_rec.size(0), x_rec.size(1), -1)
    xt = denormalize_tensor(x_true).view(x_true.size(0), x_true.size(1), -1)
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    mu_x = xr.mean(dim=2)
    mu_y = xt.mean(dim=2)
    var_x = xr.var(dim=2, unbiased=False)
    var_y = xt.var(dim=2, unbiased=False)
    cov_xy = ((xr - mu_x.unsqueeze(2)) * (xt - mu_y.unsqueeze(2))).mean(dim=2)
    ssim = ((2 * mu_x * mu_y + C1) * (2 * cov_xy + C2)) / (
        (mu_x ** 2 + mu_y ** 2 + C1) * (var_x + var_y + C2)
    )
    return float(ssim.mean().item())


def total_variation(x):
    return torch.mean(torch.abs(x[:, :, :, :-1] - x[:, :, :, 1:])) + \
           torch.mean(torch.abs(x[:, :, :-1, :] - x[:, :, 1:, :]))


def soft_cross_entropy_from_logits(logits, soft_labels):
    return torch.mean(torch.sum(-F.log_softmax(logits, dim=1) * F.softmax(soft_labels, dim=1), dim=1))


def get_parameter_grads(model, x, y=None, soft_label_logits=None, create_graph=False):
    model.zero_grad(set_to_none=True)
    logits = model(x)
    logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
    if soft_label_logits is not None:
        loss = soft_cross_entropy_from_logits(logits, soft_label_logits)
    else:
        loss = loss_fn(logits, y)
    params = [p for p in model.parameters() if p.requires_grad]
    grads = torch.autograd.grad(
        loss, params, create_graph=create_graph, retain_graph=create_graph, allow_unused=False
    )
    return grads, logits, loss


def gradient_l2_loss(current_grads, target_grads):
    loss = torch.zeros((), device=DEVICE)
    for g, tg in zip(current_grads, target_grads):
        loss = loss + torch.sum((g - tg) ** 2)
    return loss


def gradient_cosine_loss(current_grads, target_grads):
    cur = torch.cat([g.reshape(-1) for g in current_grads])
    tgt = torch.cat([tg.reshape(-1) for tg in target_grads])
    return 1.0 - F.cosine_similarity(cur, tgt, dim=0, eps=1e-12)


def infer_label_idlg(target_grads, num_classes=10):
    for g in reversed(target_grads):
        if g.dim() == 1 and g.numel() == num_classes:
            return int(torch.argmin(g).item())
    return None


def get_single_client_batch(client_id):
    loader = client_loaders[client_id]
    x, y = next(iter(loader))
    x = x[:GRAD_ATTACK_BATCH_SIZE].to(DEVICE)
    y = y[:GRAD_ATTACK_BATCH_SIZE].to(DEVICE)
    return x, y


def save_reconstruction_figure(x_true, x_rec, out_path, title):
    if not GRAD_ATTACK_SAVE_IMAGES:
        return
    xt = denormalize_tensor(x_true)[0].squeeze(0).numpy()
    xr = denormalize_tensor(x_rec)[0].squeeze(0).numpy()
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(xt, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(xr, cmap="gray")
    axes[1].set_title("Reconstructed")
    axes[1].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def run_dlg_attack(model, x_true, y_true, target_grads):
    dummy_x = torch.randn_like(x_true, requires_grad=True, device=DEVICE)
    dummy_label_logits = torch.randn((x_true.size(0), 10), requires_grad=True, device=DEVICE)
    optimizer = optim.Adam([dummy_x, dummy_label_logits], lr=GRAD_ATTACK_LR)
    for _ in range(GRAD_ATTACK_ITERS):
        optimizer.zero_grad()
        current_grads, _, _ = get_parameter_grads(
            model, dummy_x, soft_label_logits=dummy_label_logits, create_graph=True
        )
        loss = gradient_l2_loss(current_grads, target_grads)
        loss.backward()
        optimizer.step()
    pred_label = int(torch.argmax(dummy_label_logits.detach(), dim=1)[0].item())
    return dummy_x.detach(), pred_label


def run_idlg_attack(model, x_true, y_true, target_grads):
    inferred_label = infer_label_idlg(target_grads)
    if inferred_label is None:
        inferred_label = int(y_true[0].item())
    y_inferred = torch.tensor([inferred_label], device=DEVICE, dtype=torch.long)
    dummy_x = torch.randn_like(x_true, requires_grad=True, device=DEVICE)
    optimizer = optim.Adam([dummy_x], lr=GRAD_ATTACK_LR)
    for _ in range(GRAD_ATTACK_ITERS):
        optimizer.zero_grad()
        current_grads, _, _ = get_parameter_grads(model, dummy_x, y=y_inferred, create_graph=True)
        loss = gradient_l2_loss(current_grads, target_grads)
        loss.backward()
        optimizer.step()
    return dummy_x.detach(), inferred_label


def run_invgrad_attack(model, x_true, y_true, target_grads):
    inferred_label = infer_label_idlg(target_grads)
    if inferred_label is None:
        inferred_label = int(y_true[0].item())
    y_inferred = torch.tensor([inferred_label], device=DEVICE, dtype=torch.long)
    dummy_x = torch.randn_like(x_true, requires_grad=True, device=DEVICE)
    optimizer = optim.Adam([dummy_x], lr=GRAD_ATTACK_LR)
    for _ in range(GRAD_ATTACK_ITERS):
        optimizer.zero_grad()
        current_grads, _, _ = get_parameter_grads(model, dummy_x, y=y_inferred, create_graph=True)
        loss = gradient_cosine_loss(current_grads, target_grads)
        loss = loss + 1e-4 * total_variation(dummy_x) + 1e-5 * torch.mean(dummy_x ** 2)
        loss.backward()
        optimizer.step()
    return dummy_x.detach(), inferred_label


def empty_gradient_attack_results():
    return {
        "DLG MSE": None, "DLG PSNR": None, "DLG SSIM": None, "DLG Label Recovery": None,
        "iDLG MSE": None, "iDLG PSNR": None, "iDLG SSIM": None, "iDLG Label Recovery": None,
        "invGrad MSE": None, "invGrad PSNR": None, "invGrad SSIM": None, "invGrad Label Recovery": None,
        "Gradient Attack Samples": 0, "Gradient Attack CSV": None
    }


def run_gradient_inversion_attacks(model, framework_name, label, dp=False, epsilon=None):
    if not RUN_GRADIENT_ATTACKS:
        return empty_gradient_attack_results()

    model = copy.deepcopy(model).to(DEVICE)
    model.eval()
    records = []
    safe_label = label.replace(".", "p").replace("/", "_")

    for client_id in GRAD_ATTACK_CLIENTS:
        if client_id >= len(client_loaders):
            continue
        x_true, y_true = get_single_client_batch(client_id)
        if x_true.size(0) != 1:
            print("Gradient attacks expect batch size 1; skipping client", client_id)
            continue

        target_grads, _, _ = get_parameter_grads(model, x_true, y=y_true, create_graph=False)
        target_grads = [g.detach() for g in target_grads]
        true_label = int(y_true[0].item())

        for attack_name, attack_fn in [
            ("DLG", run_dlg_attack),
            ("iDLG", run_idlg_attack),
            ("invGrad", run_invgrad_attack)
        ]:
            print(f"Running {attack_name} | client {client_id} | true label {true_label}")
            x_rec, pred_label = attack_fn(model, x_true, y_true, target_grads)
            mse = tensor_mse(x_rec, x_true)
            psnr = tensor_psnr(x_rec, x_true)
            ssim = tensor_ssim_simple(x_rec, x_true)
            label_ok = int(pred_label == true_label)

            img_path = os.path.join(
                GRAD_ATTACK_DIR,
                f"{framework_name}_{DATASET_NAME}_{safe_label}_client{client_id}_{attack_name}.png"
            )
            save_reconstruction_figure(
                x_true, x_rec, img_path,
                f"{attack_name} | client {client_id} | pred={pred_label}, true={true_label}"
            )

            records.append({
                "Framework": framework_name,
                "Dataset": DATASET_NAME,
                "DP": dp,
                "Epsilon": epsilon if dp else None,
                "Setting": label,
                "Client ID": client_id,
                "Attack": attack_name,
                "True Label": true_label,
                "Predicted Label": pred_label,
                "Label Recovery": label_ok,
                "MSE": mse,
                "PSNR": psnr,
                "SSIM": ssim,
                "Image Path": img_path
            })

    if len(records) == 0:
        return empty_gradient_attack_results()

    grad_df = pd.DataFrame(records)
    grad_csv = os.path.join(
        GRAD_ATTACK_DIR,
        f"gradient_attacks_{framework_name}_{DATASET_NAME}_{safe_label}.csv"
    )
    grad_df.to_csv(grad_csv, index=False)

    out = {"Gradient Attack Samples": len(grad_df), "Gradient Attack CSV": grad_csv}
    for attack_name in ["DLG", "iDLG", "invGrad"]:
        sub = grad_df[grad_df["Attack"] == attack_name]
        out[f"{attack_name} MSE"] = float(sub["MSE"].mean()) if len(sub) else None
        out[f"{attack_name} PSNR"] = float(sub["PSNR"].mean()) if len(sub) else None
        out[f"{attack_name} SSIM"] = float(sub["SSIM"].mean()) if len(sub) else None
        out[f"{attack_name} Label Recovery"] = float(sub["Label Recovery"].mean()) if len(sub) else None

    print("Saved gradient attack CSV:", grad_csv)
    return out

# ============================================================
# RUNNERS
# ============================================================

def save_progress(round_records, progress_path):
    pd.DataFrame(round_records).to_csv(progress_path, index=False)


def run_fl(dp=False, epsilon=None):
    label = "DP" if dp else "NoDP"
    eps_label = f"_eps_{epsilon}" if dp else ""
    print(f"\n--- {DATASET_NAME} | FL-{label}{eps_label} ---")

    global_model = build_model()
    accs, times, round_records = [], [], []
    noise_total = 0.0
    progress_path = os.path.join(OUTPUT_DIR, f"progress_{DATASET_NAME}_FL_{label}{eps_label}.csv")
    start = time.time()

    for rnd in trange(ROUNDS, desc=f"[FL-{label}{eps_label}]"):
        lr = get_lr(rnd)
        local_models = []
        for node_id in range(NUM_NODES):
            local_model = train_one_client(global_model, client_loaders[node_id], lr)
            local_models.append(local_model)

        if dp:
            global_model, noise_norm = apply_fl_dp(global_model, local_models, epsilon)
            noise_total += noise_norm
        else:
            global_vec = get_vec(global_model)
            updates = [get_vec(m) - global_vec for m in local_models]
            avg_update = torch.stack(updates, dim=0).mean(dim=0)
            set_vec(global_model, global_vec + avg_update)

        acc = evaluate(global_model)
        accs.append(acc)
        times.append(time.time() - start)
        round_records.append({
            "Framework": "FL", "Dataset": DATASET_NAME, "Seed": SEED,
            "Nodes": NUM_NODES, "Round": rnd + 1, "Local Epochs": LOCAL_EPOCHS,
            "DP": dp, "Epsilon": epsilon if dp else None, "Accuracy": acc,
            "Best Accuracy So Far": max(accs), "Run Time So Far": times[-1],
            "Total Noise Norm So Far": noise_total
        })

        if rnd == 0 or (rnd + 1) % 10 == 0:
            print(f"Round {rnd + 1:03d} | LR {lr:.5f} | Acc {acc:.4f} | Best {max(accs):.4f}")
        if (rnd + 1) % SAVE_EVERY == 0 or rnd == 0 or rnd == ROUNDS - 1:
            save_progress(round_records, progress_path)
            print("Saved progress to:", progress_path)

    model_path = os.path.join(MODEL_DIR, f"{DATASET_NAME}_FL_{label}{eps_label}.pt")
    torch.save(global_model.state_dict(), model_path)

    print(f"\nRunning MIA attacks for {DATASET_NAME} FL-{label}{eps_label}...")
    attack_results = run_three_attacks(global_model)

    print(f"\nRunning gradient-inversion attacks for {DATASET_NAME} FL-{label}{eps_label}...")
    grad_results = run_gradient_inversion_attacks(
        global_model, framework_name="FL", label=f"{label}{eps_label}", dp=dp, epsilon=epsilon
    )

    row = {
        "Framework": "FL", "Dataset": DATASET_NAME, "Seed": SEED,
        "Nodes": NUM_NODES, "Rounds": ROUNDS, "Local Epochs": LOCAL_EPOCHS,
        "DP": dp, "Epsilon": epsilon if dp else None, "Threshold": THRESHOLD,
        "Final Accuracy": accs[-1], "Best Accuracy": max(accs),
        "Rounds-to-Threshold": rounds_to_threshold(accs, THRESHOLD),
        "Stability Round": stability_round(accs), "Total Noise Norm": noise_total,
        "Final Model Path": model_path,
        **attack_results, **grad_results,
        "Run Time (s)": time.time() - start
    }
    rdf = pd.DataFrame(round_records)
    rdf["Smoothed Accuracy"] = moving_average_padded(rdf["Accuracy"].tolist(), 5)
    return row, rdf


def run_dfl(dp=False, epsilon=None):
    label = "DP" if dp else "NoDP"
    eps_label = f"_eps_{epsilon}" if dp else ""
    print(f"\n--- {DATASET_NAME} | DFL-{label}{eps_label} ---")

    rng = np.random.default_rng(SEED)
    token = build_model()
    accs, times, round_records = [], [], []
    noise_total = 0.0
    progress_path = os.path.join(OUTPUT_DIR, f"progress_{DATASET_NAME}_DFL_{label}{eps_label}.csv")
    start = time.time()

    for rnd in trange(ROUNDS, desc=f"[DFL-{label}{eps_label}]"):
        lr = get_lr(rnd)
        before_vec = get_vec(token).clone()
        order = random_cover_order(NUM_NODES, rng)

        for node_id in order:
            trained_model = train_one_client(token, client_loaders[node_id], lr)
            token.load_state_dict(copy.deepcopy(trained_model.state_dict()))

        after_vec = get_vec(token).clone()
        if dp:
            token, noise_norm = apply_dfl_dp(token, before_vec, after_vec, epsilon)
            noise_total += noise_norm

        acc = evaluate(token)
        accs.append(acc)
        times.append(time.time() - start)
        round_records.append({
            "Framework": "DFL", "Dataset": DATASET_NAME, "Seed": SEED,
            "Nodes": NUM_NODES, "Round": rnd + 1, "Local Epochs": LOCAL_EPOCHS,
            "DP": dp, "Epsilon": epsilon if dp else None, "Accuracy": acc,
            "Best Accuracy So Far": max(accs), "Run Time So Far": times[-1],
            "Total Noise Norm So Far": noise_total
        })

        if rnd == 0 or (rnd + 1) % 10 == 0:
            print(f"Round {rnd + 1:03d} | LR {lr:.5f} | Acc {acc:.4f} | Best {max(accs):.4f}")
        if (rnd + 1) % SAVE_EVERY == 0 or rnd == 0 or rnd == ROUNDS - 1:
            save_progress(round_records, progress_path)
            print("Saved progress to:", progress_path)

    model_path = os.path.join(MODEL_DIR, f"{DATASET_NAME}_DFL_{label}{eps_label}.pt")
    torch.save(token.state_dict(), model_path)

    print(f"\nRunning MIA attacks for {DATASET_NAME} DFL-{label}{eps_label}...")
    attack_results = run_three_attacks(token)

    print(f"\nRunning gradient-inversion attacks for {DATASET_NAME} DFL-{label}{eps_label}...")
    grad_results = run_gradient_inversion_attacks(
        token, framework_name="DFL", label=f"{label}{eps_label}", dp=dp, epsilon=epsilon
    )

    row = {
        "Framework": "DFL", "Dataset": DATASET_NAME, "Seed": SEED,
        "Nodes": NUM_NODES, "Rounds": ROUNDS, "Local Epochs": LOCAL_EPOCHS,
        "DP": dp, "Epsilon": epsilon if dp else None, "Threshold": THRESHOLD,
        "Final Accuracy": accs[-1], "Best Accuracy": max(accs),
        "Rounds-to-Threshold": rounds_to_threshold(accs, THRESHOLD),
        "Stability Round": stability_round(accs), "Total Noise Norm": noise_total,
        "Final Model Path": model_path,
        **attack_results, **grad_results,
        "Run Time (s)": time.time() - start
    }
    rdf = pd.DataFrame(round_records)
    rdf["Smoothed Accuracy"] = moving_average_padded(rdf["Accuracy"].tolist(), 5)
    return row, rdf

# ============================================================
# MAIN
# ============================================================

all_rows = []
all_rounds = []

frameworks_to_run = ["FL", "DFL"] if FRAMEWORK == "BOTH" else [FRAMEWORK]

for fw in frameworks_to_run:
    if fw not in ["FL", "DFL"]:
        raise ValueError("FRAMEWORK must be 'FL', 'DFL', or 'BOTH'.")

    if RUN_NODP:
        row, rdf = run_fl(dp=False, epsilon=None) if fw == "FL" else run_dfl(dp=False, epsilon=None)
        all_rows.append(row)
        all_rounds.append(rdf)
        pd.DataFrame(all_rows).to_csv(os.path.join(OUTPUT_DIR, "partial_summary_MNIST.csv"), index=False)

    if RUN_DP:
        for eps in EPSILONS:
            row, rdf = run_fl(dp=True, epsilon=eps) if fw == "FL" else run_dfl(dp=True, epsilon=eps)
            all_rows.append(row)
            all_rounds.append(rdf)
            pd.DataFrame(all_rows).to_csv(os.path.join(OUTPUT_DIR, "partial_summary_MNIST.csv"), index=False)

summary_df = pd.DataFrame(all_rows)
rounds_df = pd.concat(all_rounds, ignore_index=True)

summary_path = os.path.join(OUTPUT_DIR, f"summary_MNIST_{FRAMEWORK}_{NUM_NODES}nodes_{ROUNDS}rounds.csv")
rounds_path = os.path.join(OUTPUT_DIR, f"per_round_MNIST_{FRAMEWORK}_{NUM_NODES}nodes_{ROUNDS}rounds.csv")
final_table_path = os.path.join(OUTPUT_DIR, f"FINAL_COMPARISON_TABLE_MNIST_{FRAMEWORK}_{NUM_NODES}nodes_{ROUNDS}rounds.csv")

summary_df.to_csv(summary_path, index=False)
rounds_df.to_csv(rounds_path, index=False)
summary_df.to_csv(final_table_path, index=False)

print("\n================ FINAL SUMMARY ================\n")
print(summary_df)
print("\nSaved summary to:", summary_path)
print("Saved per-round results to:", rounds_path)
print("Saved final comparison table to:", final_table_path)
print("All outputs saved in:", OUTPUT_DIR)


CUDA available: True
Using device: cuda
Dataset: MNIST
Framework: DFL
Output: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS


100%|██████████| 9.91M/9.91M [00:01<00:00, 7.78MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 219kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.04MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.51MB/s]


Saved data distribution figure: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS\figures\MNIST_data_distribution_FL_DFL_100nodes.png
Saved data distribution CSV: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS\figures\MNIST_data_distribution_FL_DFL_100nodes.csv

--- MNIST | DFL-NoDP ---


[DFL-NoDP]:   1%|          | 1/100 [00:13<22:22, 13.56s/it]

Round 001 | LR 0.01000 | Acc 0.9118 | Best 0.9118
Saved progress to: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS\progress_MNIST_DFL_NoDP.csv


...

[DFL-DP_eps_4.0]:  90%|█████████ | 90/100 [19:23<02:09, 12.94s/it]

Round 090 | LR 0.01000 | Acc 0.9893 | Best 0.9901
Saved progress to: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS\progress_MNIST_DFL_DP_eps_4.0.csv


[DFL-DP_eps_4.0]: 100%|██████████| 100/100 [21:32<00:00, 12.93s/it]

Round 100 | LR 0.01000 | Acc 0.9890 | Best 0.9910
Saved progress to: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS\progress_MNIST_DFL_DP_eps_4.0.csv

Running MIA attacks for MNIST DFL-DP_eps_4.0...


Confidence MIA | Accuracy: 0.5060 | AUC: 0.5113 | Gap: 0.002042
Loss MIA | Accuracy: 0.5090 | AUC: 0.5091 | Gap: 0.000790
Entropy MIA | Accuracy: 0.5105 | AUC: 0.5129 | Gap: 0.003117

Running gradient-inversion attacks for MNIST DFL-DP_eps_4.0...
Running DLG | client 0 | true label 1
Running iDLG | client 0 | true label 1
Running invGrad | client 0 | true label 1
Running DLG | client 10 | true label 6
Running iDLG | client 10 | true label 6
Running invGrad | client 10 | true label 6
Running DLG | client 25 | true label 1
Running iDLG | client 25 | true label 1
Running invGrad | client 25 | true label 1
Saved gradient attack CSV: FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS\gradient_attacks\gradient_attacks_DFL_MNIST_DP_eps_4p0.csv

================ FINAL SUMMARY ================

  Framework Dataset  Seed  Nodes  Rounds  Local Epochs     DP  Epsilon  \
0       DFL   MNIST     1    100     100             1  False      NaN   
1       DFL   MNIST     1    100     100             1   True     